# 🎮 Games Recommender System

A machine learning-based recommendation system that suggests games to users based on their preferences, ratings, genres, and similarity between games.

---

## 📌 Project Overview

This project focuses on building a game recommendation engine using data science and machine learning techniques. The system analyzes game-related data and generates personalized recommendations for users.

---

## 🎯 Objectives

- Clean and preprocess game data
- Perform exploratory data analysis (EDA)
- Build a recommendation model
- Recommend similar games based on user input
- Evaluate recommendation quality

---

## 🛠️ Technologies Used

- Python
- Pandas
- NumPy
- Scikit-learn
- Matplotlib
- Jupyter Notebook

---

## 📂 Project Workflow

1. Data Collection  
2. Data Cleaning  
3. Exploratory Data Analysis  
4. Feature Engineering  
5. Similarity Calculation  
6. Recommendation System Development  
7. Testing & Evaluation  

---

## 🚀 Goal

The goal of this project is to understand how recommendation systems work and apply machine learning techniques to create a practical game recommendation engine.

## 📦 1. Importing Libraries

In this section, we import all the required libraries for web scraping and data handling.

In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
print('done')


done


## 🔗 Step: API Request to Steam

We use the Steam Web API endpoint to fetch a complete list of game applications. After sending the request using `requests.get()`, we verify the response status code to confirm whether the API call was successful (200 = OK).

In [3]:
url = 'https://api.steampowered.com/ISteamApps/GetAppList/v2/'
response = requests.get(url)
print(response.status_code)

404


## ⚠️ API Request Failed (404 Error)

In this step, the request to the Steam API endpoint returned a **404 status code**, meaning the resource was not found or the endpoint is invalid/unavailable.

Because of this, we will not proceed with this API and will instead look for an alternative and more reliable game data source for our recommender system.

Next step:
- Identify a working games dataset API (e.g., RAWG or other public game databases)
- Validate the endpoint before integrating it into the project

## 🔑 Steam API Authentication Setup

The previous Steam API endpoint returned a 404 error, indicating that the resource was unavailable or required a different access method.

To proceed, we generated a Steam Web API key through the Steam Developer Portal and switched to an authenticated endpoint provided by Steam.

This API key will be used to access the application list and retrieve game-related data for the recommender system.


In [4]:
url = "https://api.steampowered.com/IStoreService/GetAppList/v1/?key=DBF71B5792579AC27797179862F3F4AC&include_games=1" #added api key from steam community
response = requests.get(url)
print(response.status_code)

200


## ✅ Successful API Connection

The authenticated request returned a **200 OK** status code, confirming that the connection to the Steam API was successful and the endpoint is working correctly.

We can now proceed with retrieving and processing the game data.

In [5]:
json_file = response.json()
print(json_file['response'].keys())

dict_keys(['apps', 'have_more_results', 'last_appid'])


In [6]:
print(json_file['response']['apps'][:3])

[{'appid': 10, 'name': 'Counter-Strike', 'last_modified': 1745368572, 'price_change_number': 34672985}, {'appid': 20, 'name': 'Team Fortress Classic', 'last_modified': 1745368565, 'price_change_number': 34672985}, {'appid': 30, 'name': 'Day of Defeat', 'last_modified': 1745368580, 'price_change_number': 34672985}]


## Steam API Pagination

The Steam API is paginated, meaning it does not return all data at once. Each response gives a batch of apps plus a `last_appid` to continue from.

---

## How the API Handles Pagination

Each response from the API contains:

- `apps` → the current batch of results  
- `have_more_results` → indicates if more data exists  
- `last_appid` → the ID used to continue from the next batch  

Example:
```json
{
  "apps": [...],
  "have_more_results": true,
  "last_appid": 507400
}

In [2]:
AppID= []
Name = []
have_more_results = "true"
last_appid= None
ogurl = 'https://api.steampowered.com/IStoreService/GetAppList/v1/?key=DBF71B5792579AC27797179862F3F4AC&include_games=1' 
while have_more_results:
 if last_appid is None:
  response = requests.get(ogurl)
 else: response = requests.get(ogurl+ f"&last_appid={last_appid}")
 data = response.json()['response']
 for app in data['apps']:
  AppID.append(app['appid'])
  Name.append(app['name'])
 last_appid = data.get('last_appid')
 have_more_results= data.get('have_more_results')
 if not have_more_results:
    print("All apps collected ✔")
    break
 print(f'collected {len(AppID)} app so far')
  

collected 10000 app so far
collected 20000 app so far
collected 30000 app so far
collected 40000 app so far
collected 50000 app so far
collected 60000 app so far
collected 70000 app so far
collected 80000 app so far
collected 90000 app so far
collected 100000 app so far
collected 110000 app so far
collected 120000 app so far
collected 130000 app so far
collected 140000 app so far
collected 150000 app so far
collected 160000 app so far
collected 170000 app so far
All apps collected ✔


## Converting Collected Data into a DataFrame

Now that all Steam apps have been successfully collected using the API loop, the next step is to structure the data for analysis.

We currently have two lists:
- `AppID` → contains all app IDs
- `Name` → contains all corresponding game names

### Next Step

We will convert these lists into a pandas DataFrame to make the data easier to work with:

- Organize data into rows and columns
- Prepare it for analysis and filtering
- Set up the dataset for future recommendation system work

The final structured format will look like:

| Game_ID | Game_Name |
|------|------|
| 10   | Counter-Strike |
| 20   | Team Fortress Classic |
| ...  | ... |

In [15]:
steam_df = pd.DataFrame(
    {
        "Game_ID": AppID,
        "Game_Name": Name
    }
)
steam_df.head()

,Game_ID,Game_Name
0,10,Counter-Strike
1,20,Team Fortress Classic
2,30,Day of Defeat
3,40,Deathmatch Classic
4,50,Half-Life: Opposing Force


In [17]:
steam_df.to_csv(r"D:\Projects\games-recommender-system\data\raw\steam_games.csv", index=False)

## Scraping Detailed Steam Game Data

Now that we successfully collected the Steam app list, the next step is to scrape detailed information for each game using the Steam Store API.

We will use the following endpoint format:

https://store.steampowered.com/api/appdetails?appids=10

In this API:
- `appids=10` represents the Steam application ID
- The API returns detailed metadata about the game
- The response is returned as a JSON object

From this endpoint, we can extract useful features such as:
- If Game is free
- Short Description
- Supported Languages
- Developers
- Publishers
- Price information
- Genres
- Metacritic score
- Categories
- Recommendations
- Release Date

These features will later help us during:
- Data cleaning
- Exploratory Data Analysis (EDA)
- Feature engineering
- Building the recommendation system

In [ ]:
meta_url = 'https://store.steampowered.com/api/appdetails?appids=10&l=english'
meta_response = requests.get(meta_url)
meta_data = meta_response.json()
print(meta_data['10']['data'].keys())

dict_keys(['type', 'name', 'steam_appid', 'required_age', 'is_free', 'detailed_description', 'about_the_game', 'short_description', 'supported_languages', 'header_image', 'capsule_image', 'capsule_imagev5', 'website', 'pc_requirements', 'mac_requirements', 'linux_requirements', 'developers', 'publishers', 'price_overview', 'packages', 'package_groups', 'platforms', 'metacritic', 'categories', 'genres', 'screenshots', 'recommendations', 'release_date', 'support_info', 'background', 'background_raw', 'content_descriptors', 'ratings'])


In [ ]:
meta_id = []
is_free = []
descriptions = []
languages = []
developers = []
publishers = []
price = []
genre = []
metacritic = []
catergory = []
recommendations = []
release_date =[]
apps_collected = 0
pd.DataFrame(columns=["Game_ID", "Is_Free", "Description", "Supported_Languages", 
                       "Developer", "Publishers", "Price", "Genre", "Metacritic_Score",
                       "Category", "Number_of_Recommendations", "Release_Date"]
).to_csv(r"D:\Projects\games-recommender-system\data\raw\meta_data.csv", index=False) #initializing the metadata file as empty csv
for app in AppID:
    if apps_collected >= 10000: #breaks the loop when we have more than 10k games
        break
    else:
        meta_response = requests.get('https://store.steampowered.com/api/appdetails?appids='+ f"{app}" +'&l=english')
        response_data = meta_response.json()[f'{app}']
        if not response_data.get('success'):
            continue #skips the app/game if the data return is not successful
        meta_data = response_data['data']
        if meta_data['type'] =='game':
            meta_id.append(meta_data.get('steam_appid', None)) #returns None in case the value was not found to avoid breaking loop
            genre.append(meta_data.get('genres', None))
            is_free.append(meta_data.get('is_free', None))
            descriptions.append(meta_data.get('short_description', None))
            languages.append(meta_data.get('supported_languages', None))
            developers.append(meta_data.get('developers', None))
            publishers.append(meta_data.get('publishers', None))
            price.append(meta_data.get('price_overview', {}).get('final_formatted', None)) 
            metacritic.append(meta_data.get('metacritic',{}).get('score', None))
            catergory.append(meta_data.get('categories', None))
            recommendations.append(meta_data.get('recommendations', {}).get('total', None))
            release_date.append(meta_data.get('release_date', {}).get('date', None))
            print(f"Collected {apps_collected} games so far...")
            apps_collected +=1
            if len(meta_id) % 500 == 0:
                meta_df = pd.DataFrame(
                {
                    "Game_ID": meta_id,
                    "Is_Free": is_free,
                    "Description": descriptions,
                    "Supported_Languages": languages,
                    "Developer": developers,
                    "Publishers": publishers,
                    "Price": price,
                    "Genre": genre,
                    "Metacritic_Score": metacritic,
                    "Category": catergory,
                    "Number_of_Recommendations": recommendations,
                    "Release_Date": release_date
                }
                )
                meta_df.to_csv(r"D:\Projects\games-recommender-system\data\raw\meta_data.csv", mode= 'a', index= False, header=False)
                meta_id.clear()
                is_free.clear()
                descriptions.clear()
                languages.clear()
                developers.clear()
                publishers.clear()
                price.clear()
                genre.clear()
                metacritic.clear()
                catergory.clear()
                recommendations.clear()
                release_date.clear()
            time.sleep(1.5)
            continue
        
            
print('all meta data were collected')

Collected 1 games so far...
Collected 2 games so far...
Collected 3 games so far...
Collected 4 games so far...
Collected 5 games so far...
Collected 6 games so far...
Collected 7 games so far...
Collected 8 games so far...
Collected 9 games so far...
Collected 10 games so far...
Collected 11 games so far...
Collected 12 games so far...
Collected 13 games so far...
Collected 14 games so far...
Collected 15 games so far...
Collected 16 games so far...
Collected 17 games so far...
Collected 18 games so far...
Collected 19 games so far...
Collected 20 games so far...
Collected 21 games so far...
Collected 22 games so far...
Collected 23 games so far...
Collected 24 games so far...
Collected 25 games so far...
Collected 26 games so far...
Collected 27 games so far...
Collected 28 games so far...
Collected 29 games so far...
Collected 30 games so far...
Collected 31 games so far...
Collected 32 games so far...
Collected 33 games so far...
Collected 34 games so far...
Collected 35 games so f

PermissionError: [Errno 13] Permission denied: 'D:\\Projects\\games-recommender-system\\data\\raw\\meta_data.csv'

### Resuming Metadata Collection

The scraping process stopped because the `meta_data.csv` file was open while Python attempted to append new data to it, causing a `PermissionError`.

To continue the collection process without losing progress, we will restart the scraper beginning from the last successfully processed App ID and continue appending the remaining game metadata to the existing CSV file.

In [8]:
AppID.index(72900)

1022

In [10]:
apps_collected = 1000
for app in AppID[1023:]:
    if apps_collected >= 10000: #breaks the loop when we have more than 10k games
        break
    else:
        meta_response = requests.get('https://store.steampowered.com/api/appdetails?appids='+ f"{app}" +'&l=english')
        response_data = meta_response.json()[f'{app}']
        if not response_data.get('success'):
            continue #skips the app/game if the data return is not successful
        meta_data = response_data['data']
        if meta_data['type'] =='game':
            meta_id.append(meta_data.get('steam_appid', None)) #returns None in case the value was not found to avoid breaking loop
            genre.append(meta_data.get('genres', None))
            is_free.append(meta_data.get('is_free', None))
            descriptions.append(meta_data.get('short_description', None))
            languages.append(meta_data.get('supported_languages', None))
            developers.append(meta_data.get('developers', None))
            publishers.append(meta_data.get('publishers', None))
            price.append(meta_data.get('price_overview', {}).get('final_formatted', None)) 
            metacritic.append(meta_data.get('metacritic',{}).get('score', None))
            catergory.append(meta_data.get('categories', None))
            recommendations.append(meta_data.get('recommendations', {}).get('total', None))
            release_date.append(meta_data.get('release_date', {}).get('date', None))
            print(f"Collected {apps_collected} games so far...")
            apps_collected +=1
            if len(meta_id) % 500 == 0:
                meta_df = pd.DataFrame(
                {
                    "Game_ID": meta_id,
                    "Is_Free": is_free,
                    "Description": descriptions,
                    "Supported_Languages": languages,
                    "Developer": developers,
                    "Publishers": publishers,
                    "Price": price,
                    "Genre": genre,
                    "Metacritic_Score": metacritic,
                    "Category": catergory,
                    "Number_of_Recommendations": recommendations,
                    "Release_Date": release_date
                }
                )
                meta_df.to_csv(r"D:\Projects\games-recommender-system\data\raw\meta_data.csv", mode= 'a', index= False, header=False)
                meta_id.clear()
                is_free.clear()
                descriptions.clear()
                languages.clear()
                developers.clear()
                publishers.clear()
                price.clear()
                genre.clear()
                metacritic.clear()
                catergory.clear()
                recommendations.clear()
                release_date.clear()
            time.sleep(1.5)
            continue
        
            
print('all meta data were collected')

Collected 1000 games so far...
Collected 1001 games so far...
Collected 1002 games so far...
Collected 1003 games so far...
Collected 1004 games so far...
Collected 1005 games so far...
Collected 1006 games so far...
Collected 1007 games so far...
Collected 1008 games so far...
Collected 1009 games so far...
Collected 1010 games so far...
Collected 1011 games so far...
Collected 1012 games so far...
Collected 1013 games so far...
Collected 1014 games so far...
Collected 1015 games so far...
Collected 1016 games so far...
Collected 1017 games so far...
Collected 1018 games so far...
Collected 1019 games so far...
Collected 1020 games so far...
Collected 1021 games so far...
Collected 1022 games so far...
Collected 1023 games so far...
Collected 1024 games so far...
Collected 1025 games so far...
Collected 1026 games so far...
Collected 1027 games so far...
Collected 1028 games so far...
Collected 1029 games so far...
Collected 1030 games so far...
Collected 1031 games so far...
Collecte

In [12]:
meta_df = pd.read_csv(r"D:\Projects\games-recommender-system\data\raw\meta_data.csv")
print(meta_df.shape)
print(meta_df.head())

(10500, 12)
   Game_ID  Is_Free                                        Description  \
0       10    False  Play the world's number 1 online action game. ...   
1       20    False  One of the most popular online action games of...   
2       30    False  Enlist in an intense brand of Axis vs. Allied ...   
3       40    False  Enjoy fast-paced multiplayer gaming with Death...   
4       50    False  Return to the Black Mesa Research Facility as ...   

                                 Supported_Languages             Developer  \
0  English<strong>*</strong>, French<strong>*</st...             ['Valve']   
1  English, French, German, Italian, Spanish - Sp...             ['Valve']   
2  English, French, German, Italian, Spanish - Spain             ['Valve']   
3  English, French, German, Italian, Spanish - Sp...             ['Valve']   
4                    English, French, German, Korean  ['Gearbox Software']   

  Publishers      Price                                   Genre  \
0  ['Va

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive",
}
db_response = requests.get('https://steamdb.info/app/4000/', headers= headers)
print(db_response.status_code)

403


## SteamDB Scraping Strategy Update

We attempted to access SteamDB using `requests` with browser-like headers, but the server returns a **403 Forbidden** response.

### Reason
SteamDB is protected by **Cloudflare bot protection**, which blocks:
- Non-browser HTTP clients (e.g., `requests`)
- Missing browser fingerprinting (TLS/JS execution)
- Automated scraping patterns

### Decision
Since direct HTTP requests are blocked, we will switch to a browser automation approach.

### New Approach
We will use **Selenium** to simulate a real browser session, which allows:
- Full JavaScript rendering
- Proper Cloudflare challenge handling
- Access to dynamically loaded data (tables, stats, etc.)

### Why Selenium
- Executes JavaScript like a real user
- Maintains cookies/session automatically
- More reliable for SteamDB scraping compared to `requests` + BeautifulSoup

### Expected Outcome
Using Selenium should allow us to extract:
- Current players
- 24h peak players
- Followers
- Owner estimates
- Tags
- Price information (where available)

### Next Step
Implement Selenium-based scraper and validate access to SteamDB pages before extracting structured data.

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")


In [10]:
soup = BeautifulSoup(driver.page_source, 'html.parser')

In [ ]:
#to get the current player
current_players = soup.find_all(class_='header-thing-number')[1].text
current_players

'23,193'

In [ ]:
#to get number of reviews 
number_of_reviews = soup.find_all(class_='header-thing header-thing-good tooltipped tooltipped-n')
value_count = number_of_reviews[0].find('meta', itemprop= 'reviewCount')['content']
value_count


'1245243'

In [ ]:
#to get positive review percentage
reviews_percentage = soup.find_all(class_='header-thing header-thing-good tooltipped tooltipped-n')
positive_reviews_percentage = reviews_percentage[0].find('meta', itemprop= 'ratingValue')['content']
positive_reviews_percentage

'96.18'

In [42]:
#and finally to get tags 
tags = soup.find_all(class_='store-tags')
game_tags = [tag.text for tag in tags[0].find_all('a')]
game_tags

['🧱 Sandbox',
 '🛠️ Moddable',
 '🕹️ Multiplayer',
 '⚛️ Physics',
 '🏗️ Building',
 '😌 Casual',
 '😃 Funny',
 '🙌 First-Person',
 '👤 Singleplayer',
 '🔫 FPS',
 '🖥️ Simulation',
 '🤡 Comedy',
 '👥 Online Co-Op',
 '🔫 Shooter',
 '🤝 Co-op',
 '💥 Action',
 '🌱 Indie',
 '🥊 PvP',
 '📷 Realistic',
 '🗺 Exploration']

In [3]:
meta_df = pd.read_csv(r"D:\Projects\games-recommender-system\data\raw\meta_data.csv")
Game_ID = meta_df['Game_ID'].tolist()

In [21]:
import random
game_tags = []
online_players = []
total_reviews = []
positive_reviews_percentage = []
game_id = []
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
pd.DataFrame(columns=["Game_ID","Current Players", "Total Reviews", "Positive Reviews Percentage", "Tags"]).to_csv(r"D:\Projects\games-recommender-system\data\raw\DB_Meta.csv", index= False)
driver.get('https://steamdb.info/app/10')
time.sleep(20)
for game in Game_ID:

    driver.get('https://steamdb.info/app/'+ f"{game}")

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    tag_selection = soup.find_all(class_='store-tags')
    extracted_tags = [tag.text for tag in tag_selection[0].find_all('a')] if tag_selection else None
    game_tags.append(extracted_tags)

    reviews_percentage = soup.find_all(class_='header-thing header-thing-good tooltipped tooltipped-n')
    prp = reviews_percentage[0].find('meta', itemprop= 'ratingValue')['content'] if reviews_percentage else None
    positive_reviews_percentage.append(prp)

    number_of_reviews = soup.find_all(class_='header-thing header-thing-good tooltipped tooltipped-n')
    value_count = number_of_reviews[0].find('meta', itemprop= 'reviewCount')['content'] if number_of_reviews else None
    total_reviews.append(value_count)

    players_elements = soup.find_all(class_='header-thing-number')
    current_players = players_elements[1].text if len(players_elements) > 1 else None
    online_players.append(current_players)

    game_id.append(game)
    if len(game_id) % 200 == 0:
        driver.quit()
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        time.sleep(10)
        db_meta = pd.DataFrame(
            {
                "Game_ID": game_id,
                "Tags": game_tags,
                "Total Reviews": total_reviews,
                "Positive Reviews Percentage": positive_reviews_percentage,
                "Current Players": online_players
            }
        )
        db_meta.to_csv(r"D:\Projects\games-recommender-system\data\raw\DB_Meta.csv", mode= 'a', index= False, header=False)
        game_id.clear()
        game_tags.clear()
        total_reviews.clear()
        positive_reviews_percentage.clear()
        online_players.clear()
        
    time.sleep(random.uniform(3, 7))

print("all steamdb data was collected")




InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=149.0.7827.115); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff772ec3fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff772ec4000+14980]
	chromedriver!(No symbol) [0x7ff772a0793d]
	chromedriver!(No symbol) [0x7ff7729f2d82]
	chromedriver!(No symbol) [0x7ff772a190cc]
	chromedriver!(No symbol) [0x7ff772a91aa0]
	chromedriver!(No symbol) [0x7ff772aaec82]
	chromedriver!(No symbol) [0x7ff772a5401c]
	chromedriver!(No symbol) [0x7ff772a54f43]
	chromedriver!GetHandleVerifier [0x7ff7734a7591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff7734a1902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff7734c7115+617a95]
	chromedriver!GetHandleVerifier [0x7ff772ee1dce+3274e]
	chromedri

## Dataset Consolidation

Originally, the project plan included collecting additional game information by scraping SteamDB to enrich our existing dataset. However, due to technical limitations and repeated rate-limiting issues encountered during the scraping process, obtaining the remaining data from SteamDB was not feasible.

After evaluating the current state of the project, we determined that the data already collected is sufficient to proceed with the development of our game recommendation system. Instead of relying on additional scraping, we will merge the two existing datasets into a single comprehensive dataset using the shared `Game_ID` field.

The first dataset contains the basic game identification information:

| Column |
|--------|
| `Game_ID` |
| `Game_Name` |

The second dataset contains the detailed game metadata:

| Column |
|--------|
| `Game_ID` |
| `Is_Free` |
| `Description` |
| `Supported_Languages` |
| `Developer` |
| `Publishers` |
| `Price` |
| `Genre` |
| `Metacritic_Score` |
| `Category` |
| `Number_of_Recommendations` |
| `Release_Date` |

By performing an inner join on the `Game_ID` column, we will obtain a unified dataset containing both the game names and their associated metadata. This consolidated dataset provides all the essential information required for the subsequent stages of preprocessing, feature engineering, exploratory data analysis, and building the recommendation engine.

In [4]:
meta_data = pd.read_csv(r'D:\Projects\games-recommender-system\data\raw\meta_data.csv')
steam_games = pd.read_csv(r'D:\Projects\games-recommender-system\data\raw\steam_games.csv')
Final_Dataset = pd.merge(steam_games , meta_data, on= 'Game_ID')

In [5]:
Final_Dataset.to_csv(r'D:\Projects\games-recommender-system\data\raw\Final_Dataset.csv', index= False)

In [9]:
print(Final_Dataset.shape)
print(Final_Dataset.head())

(10496, 13)
   Game_ID                  Game_Name  Is_Free  \
0       10             Counter-Strike    False   
1       20      Team Fortress Classic    False   
2       30              Day of Defeat    False   
3       40         Deathmatch Classic    False   
4       50  Half-Life: Opposing Force    False   

                                         Description  \
0  Play the world's number 1 online action game. ...   
1  One of the most popular online action games of...   
2  Enlist in an intense brand of Axis vs. Allied ...   
3  Enjoy fast-paced multiplayer gaming with Death...   
4  Return to the Black Mesa Research Facility as ...   

                                 Supported_Languages             Developer  \
0  English<strong>*</strong>, French<strong>*</st...             ['Valve']   
1  English, French, German, Italian, Spanish - Sp...             ['Valve']   
2  English, French, German, Italian, Spanish - Spain             ['Valve']   
3  English, French, German, Italian, S

In [10]:
Final_Dataset.to_csv(r'D:\Projects\games-recommender-system\data\processed\Final_Dataset.csv', index= False)